# Model exploration: predicting next-gameweek FPL points

Compares candidate models for the production predictor on a time-based split of the
historical data. The feature engineering below mirrors `prediction/predictor.py`
(rolling means over 1/3/5/7 gameweeks plus fixture context) so the comparison
transfers directly to production.

Models: naive baseline (5-GW rolling mean), Ridge, RandomForest, XGBoost, LightGBM.

In [1]:
# Notebook-only extras (LightGBM is not a runtime dependency)
%pip install -q lightgbm xgboost scikit-learn pandas numpy


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

# Feature engineering copied from prediction/predictor.py train_model()
ROLLING_WINDOWS = [1, 3, 5, 7]
BASE_ROLLING_FEATURES = [
    'bps', 'ict_index', 'minutes', 'total_points', 'selected',
    'transfers_balance', 'now_cost', 'goals_scored', 'assists',
    'clean_sheets', 'goals_conceded', 'expected_goals',
    'expected_goal_involvements', 'expected_assists',
    'expected_goals_conceded', 'form',
]

df = pd.read_csv('../data/cleaned_merged_seasons.csv', low_memory=False)
seasons = sorted(df['season'].dropna().unique())
df = df[df['season'].isin(seasons[-3:])]  # last 3 seasons keeps runtime sane
df = df.sort_values(['name', 'season', 'GW']).copy()

grouped = df.groupby('name', sort=False)
feature_cols = []
for col in BASE_ROLLING_FEATURES:
    for window in ROLLING_WINDOWS:
        name = f"{col}_rolling_{window}"
        df[name] = grouped[col].transform(lambda s: pd.to_numeric(s, errors='coerce').rolling(window, min_periods=1).mean())
        feature_cols.append(name)

df['target'] = grouped['total_points'].shift(-1)
df['team_id_feature'] = pd.to_numeric(df.get('team_id', 0), errors='coerce').fillna(0)
df['opponent_team_feature'] = pd.to_numeric(df.get('opponent_team', 0), errors='coerce').fillna(0)
df['is_home_feature'] = pd.to_numeric(df.get('was_home', 0), errors='coerce').fillna(0)
feature_cols += ['team_id_feature', 'opponent_team_feature', 'is_home_feature']

data = df.dropna(subset=feature_cols + ['target'])
print(f"{len(data):,} rows, {len(feature_cols)} features, seasons: {seasons[-3:]}")

23,004 rows, 67 features, seasons: ['2023-24', '2024-25', '2025-26']


In [3]:
# Time-based split: last 5 GWs of the newest season held out as test
newest = seasons[-1]
data = data.copy()
data['GW_num'] = pd.to_numeric(data['GW'], errors='coerce')
last_gw = data.loc[data['season'] == newest, 'GW_num'].max()
test_mask = (data['season'] == newest) & (data['GW_num'] > last_gw - 5)

train, test = data[~test_mask], data[test_mask]
X_train = train[feature_cols].astype(np.float32)
y_train = train['target'].astype(np.float32)
X_test = test[feature_cols].astype(np.float32)
y_test = test['target'].astype(np.float32)
print(f"train: {len(train):,}   test: {len(test):,} (GW {last_gw-4:.0f}-{last_gw:.0f} of {newest})")

train: 18,990   test: 4,014 (GW 26-30 of 2025-26)


In [4]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import lightgbm as lgb

models = {
    'Naive (5-GW rolling mean)': None,
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42),
    'XGBoost': xgb.XGBRegressor(max_depth=6, learning_rate=0.1, n_estimators=150,
                                random_state=42, n_jobs=-1, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(num_leaves=63, learning_rate=0.1, n_estimators=150,
                                  random_state=42, n_jobs=-1, verbose=-1),
}

results, preds = {}, {}
for name, model in models.items():
    if model is None:
        y_pred = X_test['total_points_rolling_5'].values
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    preds[name] = y_pred
    results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_test, y_pred))),
    }

pd.DataFrame(results).T.sort_values('MAE').round(3)

,MAE,RMSE
RandomForest,0.921,1.839
Ridge,0.925,1.792
XGBoost,0.926,1.857
LightGBM,0.932,1.885
Naive (5-GW rolling mean),0.960,1.976


In [5]:
# Error broken down by position
pos = test['position'].astype(str).str.upper().reset_index(drop=True)
per_position = pd.DataFrame({
    name: pd.Series(np.abs(y_test.values - y_pred)).groupby(pos).mean()
    for name, y_pred in preds.items()
})
per_position.round(3)

,Naive (5-GW rolling mean),Ridge,RandomForest,XGBoost,LightGBM
position,,,,,
DEF,1.167,1.093,1.121,1.114,1.113
FWD,1.092,1.050,1.063,1.074,1.067
GK,0.621,0.593,0.587,0.592,0.561
MID,0.864,0.858,0.828,0.840,0.862


In [6]:
best = min(results, key=lambda k: results[k]['MAE'])
print('Best model by MAE:', best)

# Feature importances of the production model (XGBoost)
importances = pd.Series(models['XGBoost'].feature_importances_, index=feature_cols)
importances.sort_values(ascending=False).head(15).round(4)

Best model by MAE: RandomForest


minutes_rolling_1                       0.3292
minutes_rolling_3                       0.0423
now_cost_rolling_1                      0.0231
form_rolling_1                          0.0161
selected_rolling_7                      0.0135
expected_goals_conceded_rolling_1       0.0134
opponent_team_feature                   0.0131
total_points_rolling_3                  0.0130
goals_scored_rolling_1                  0.0129
is_home_feature                         0.0129
expected_goals_conceded_rolling_3       0.0126
goals_scored_rolling_3                  0.0125
expected_goals_rolling_5                0.0125
expected_goal_involvements_rolling_7    0.0124
expected_goal_involvements_rolling_3    0.0124
dtype: float32

## Conclusion

On this split (test = GW26-30 of 2025-26) every learned model clearly beats the
naive rolling-mean baseline, but RandomForest, Ridge, and XGBoost land within
~0.005 MAE of each other — statistically noise. The signal is dominated by a few
features (recent minutes above all), which is why even Ridge keeps up.

Production keeps **XGBoost**: it's already a runtime dependency, trains in seconds,
and sits within noise of the leader. Re-run this notebook as more of the current
season accumulates — if another model opens a consistent, repeatable gap across
several test windows, swapping the production model is a one-line change in
`prediction/predictor.py`.